# 🎯 [K15_M06] 반복 고장 찾기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m06_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 06. 반복 고장 찾기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 06. 반복 고장 찾기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>동일한 시설 및 설비에서 2회 이상 반복 발생하는 고장 증상을 AI를 통해 식별하고 유지보수 주의 대상을 선별합니다. 반복 고장 건의 발생 일자와 조치 소요 기간을 일목요연한 표로 정리하며, 고장 원인을 자의적으로 단정하지 않고 추가 현장 확인 항목을 함께 도출합니다. 데이터 기반의 설비 예방 관리 및 객관적 상태 분석 관점을 배양합니다.</p>

**[학습 목표 및 체득 역량]**
<p>동일 시설·설비에서 2회 이상 발생한 반복 고장을 AI로 탐색하여 이력 표를 구성하고, 단정적 판단 대신 현장 추가 확인 사항을 정리합니다. <strong>이를 통해 반복 결함 패턴을 탐지하여 예방 정비 소요를 파악하고, AI 분석 결과를 바탕으로 객관적 후속 조치를 기획하는 데이터 분석적 사고를 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: 같은 시설·같은 설비 2회 이상을 찾는다.
<h4>이 단계의 역할</h4><p>22건의 이력 데이터에서 동일한 시설의 동일한 설비(증상)가 2회 이상 반복 보고된 취약 설비 패턴을 AI로 탐색합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>AI에게 '동일 시설에서 동일 설비 또는 유사 증상으로 접수된 고장신고 건'을 묶어서 추출하도록 지시합니다.</p></li>
<li><p>세미나실 B 빔프로젝터(H01, H06, H16), 체력단련실 러닝머신 1호(H03, H11, H20), 강당 무선 마이크(H05, H13, H21)의 3대 핵심 반복 고장을 찾아냅니다.</p></li>
<li><p>세미나실 A 난방기 소음 2회(H04, H14) 등 추가 발생 건을 함께 식별합니다.</p></li>
</ol>
<h4>💡 AI 반복 고장 탐색 프롬프트 예시</h4><blockquote><p><strong>[반복 결함 탐색 질의]:</strong> "아래 점검 이력 22건에서 동일한 시설에서 동일한 장비나 증상으로 2회 이상 발생한 고장 건들을 그룹화해 줘. 각 그룹마다 발생 번호(H번호), 발생 일자, 구체적 증상을 묶어서 정리해 줘: [점검 이력 데이터]"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>3대 반복 고장 적출:</strong> 빔프로젝터 3회, 러닝머신 1호 3회, 무선 마이크 3회가 빠짐없이 적출되었는가?</li>
<li><strong>오탐 방지:</strong> 서로 다른 시설의 유사 증상(예: 강당 음향 점검과 세미나실 영상 점검)을 반복 고장으로 오인하지 않았는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>반복 고장 주요 3건 및 추가 1건 정확 식별</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>반복 고장 설비 1차 탐색 목록</p></li></ul>

#### 🔹 Step 2: 발생일과 소요일을 표로 만든다.
<h4>이 단계의 역할</h4><p>식별된 반복 고장 설비별로 발생 일자, 조치 소요일, 재발 주기(간격)를 한눈에 볼 수 있는 '반복 고장 이력표'를 제작합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>각 반복 고장 건마다 1차 발생일, 2차 발생일, 3차 발생일을 타임라인 순으로 정리합니다.</p></li>
<li><p>1차 발생부터 2차 발생까지 걸린 일수(재발 간격)와 조치 소요일을 표에 기재합니다.</p></li>
<li><p>최근 발생한 건이 여전히 '처리 중'이거나 '접수' 상태인지 확인하여 현재 진행 상황을 명시합니다.</p></li>
</ol>
<h4>💡 AI 타임라인 표 생성 예시</h4><blockquote><p><strong>[반복 고장 타임라인 질의]:</strong> "세미나실 B 빔프로젝터, 체력단련실 러닝머신 1호, 강당 무선 마이크의 고장 발생 일자와 소요일을 시간순 테이블로 만들고, 각 고장 간 발생 간격(일)을 계산해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>일자 정확도:</strong> 빔프로젝터(08-03, 09-02, 10-08) 등 발생 일자가 원문과 일치하는가?</li>
<li><strong>상태 표기:</strong> 3차 발생 건(H16 완료, H20 처리 중, H21 접수)의 상태가 정확히 표기되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>반복 고장 3대 설비 타임라인 및 소요일 정리표 완성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>반복 고장 상세 타임라인 표</p></li></ul>

#### 🔹 Step 3: 확인할 사항을 적는다.
<h4>이 단계의 역할</h4><p>반복 고장의 원인을 AI나 실무자가 임의로 예단하지 않고, 객관적 사실에 근거하여 현장에서 추가로 확인해야 할 점검 체크리스트를 도출합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>고장 원인을 '부품 불량'이나 '정비 미숙'으로 자의적으로 단정하지 않는 중립적 보고 태도를 정립합니다.</p></li>
<li><p>빔프로젝터(케이블 접촉 불량 여부, 램프 수명), 러닝머신(모터 과열, 벨트 장력), 마이크(수신기 안테나, 주파수 혼선) 등 실제 현장 확인 항목을 정리합니다.</p></li>
<li><p>부대 시설 관리관에게 보고할 '설비 상태 확인 메모'를 작성합니다.</p></li>
</ol>
<h4>💡 AI 현장 점검 항목 도출 예시</h4><blockquote><p><strong>[체크리스트 생성 질의]:</strong> "러닝머신 벨트 이탈이 3회 반복되고 무선 마이크 수신 불량이 3회 반복되었다. 원인을 섣불리 단정하지 않고, 군수과 및 통신관이 현장에서 기술적으로 점검해야 할 객관적 확인 사항 3가지씩을 제안해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>단정적 표현 배제:</strong> '부품 교체 필요'처럼 확정 짓지 않고 '현장 장력 확인 필요' 등 점검적 어조를 유지했는가?</li>
<li><strong>현장 실효성:</strong> 부대 실무에서 기술 부서가 즉시 확인할 수 있는 항목인가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>반복 고장 설비별 현장 추가 확인 사항 메모 작성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>현장 확인 체크리스트 및 관리관 보고 메모</p></li></ul>

#### 🔹 Step 4: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>반복 발생 주기가 회차를 거듭할수록 짧아지는지(고장 빈도 증가 추세)를 분석하는 심화 도전 과제를 수행합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>1차~2차 간격과 2차~3차 간격을 비교하여 고장 빈도가 가속화되는 설비를 파악합니다.</p></li>
<li><p>예를 들어 세미나실 B 빔프로젝터의 간격 변화(30일 -> 36일)와 러닝머신의 간격 변화를 분석합니다.</p></li>
<li><p>재발 주기가 짧아지는 설비에 대해 예방 정비 우선순위를 부여하는 근거를 정리합니다.</p></li>
</ol>
<h4>💡 AI 재발 주기 추세 분석 질의 예시</h4><blockquote><p><strong>[빈도 가속화 분석 질의]:</strong> "반복 고장 설비 3종의 발생 간격 추이를 분석하여, 고장 발생 주기가 점점 짧아져 전면 교체나 정밀 정비가 가장 시급한 취약 설비를 데이터 근거로 제시해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>수치 계산 정확도:</strong> 날짜 계산(일수 차이)에 오차가 없는가?</li>
<li><strong>논리적 연계:</strong> 도출된 결론이 단순 추측이 아닌 재발 일수 데이터에 기반하는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>고장 주기 가속화 분석 메모 및 예방 정비 우선순위 제안</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>심화 재발 주기 분석 보고서</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 3대 주요 반복 고장 설비 타임라인 표
| 시설 및 설비 | 1차 발생 (일자/소요) | 2차 발생 (일자/소요) | 3차 발생 (일자/소요) | 재발 간격 추이 |
| :--- | :--- | :--- | :--- | :--- |
| 세미나실 B 빔프로젝터 | H01 (08-03, 2일) | H06 (09-02, 3일) | H16 (10-08, 4일) | 30일 -> 36일 |
| 체력단련실 러닝머신 1호 | H03 (08-17, 4일) | H11 (09-21, 6일) | H20 (10-21, 처리중) | 35일 -> 30일 (단축) |
| 대대 강당 무선 마이크 | H05 (08-31, 2일) | H13 (09-28, 3일) | H21 (10-22, 접수) | 28일 -> 24일 (단축) |

## 2. 설비별 현장 추가 점검 체크리스트 (원인 예단 금지)
| 설비명 | 섣부른 추측 (지양) | 현장 객관적 점검 사항 (지향) | 담당 부서 |
| :--- | :--- | :--- | :--- |
| 빔프로젝터 | "기기가 낡아서 고장남" | 1. 케이블 단자 접촉 상태<br>2. 램프 사용 시간 누적량<br>3. 전원선 과열 여부 | 군수과 |
| 러닝머신 1호 | "장병들이 험하게 씀" | 1. 벨트 좌우 장력 균형<br>2. 모터 윤활유(오일) 상태<br>3. 롤러 베어링 마모도 | 군수과 |
| 무선 마이크 | "배터리 충전 불량" | 1. 수신기 안테나 각도/위치<br>2. 주파수 혼선 여부<br>3. 송신기 캡슐 단선 여부 | 정보통신실 |

## 3. 심화 분석: 고장 주기 가속화 분석 메모
- **강당 무선 마이크**: 재발 주기가 28일에서 24일로 점점 짧아지고 있으며, 현재 10-22 접수 건(H21)이 미완료 상태임. 수신기 교체 또는 주파수 재설정 등 근본 정비 필요.

### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] 반복 결함 탐지·객관적 분석 — 2회 이상 반복 고장 설비를 식별하고 원인을 예단하지 않는 객관적 현장 점검 리스트를 도출했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>3대 반복 고장(빔프로젝터, 러닝머신 1호, 무선 마이크) 타임라인 완성</p></li><li><p>주관적 원인 예단 배제 및 기술 부서용 현장 점검 체크리스트 도출</p></li><li><p>교육용 가상 데이터 활용 및 군사보안 수칙 엄수</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>데이터에 기반해 반복 고장을 객관적으로 보고하되, 원인을 자의적으로 단정하지 않고 전문 기술 부서의 현장 확인 항목을 도출하는 실무 분석 자세를 평가한다.</p></blockquote>


In [ ]:
# [미션 06] 프롬프트 및 실습 결과 기록
mission_06_prompt = """# [미션 06] 2회 이상 반복 고장 탐색 및 현장 객관적 점검 리스트 도출
너는 군 시설 예방 정비 분석관이다.
점검 이력 22건에서 동일 시설의 동일 설비가 2회 이상 고장 신고된 반복 결함 패턴을 탐색하라.
각 설비별 발생 일자, 조치 소요일, 재발 주기 간격을 정리하고,
원인을 임의로 예단하지 않는 객관적인 현장 점검 확인 항목 3가지를 도출하라.
"""

mission_06_result = """### 📋 반복 고장 타임라인 및 객관적 현장 확인서

#### 1. 3대 주요 반복 고장 설비 타임라인 표
| 시설 및 설비 | 1차 발생 (일자/소요) | 2차 발생 (일자/소요) | 3차 발생 (일자/소요) | 재발 간격 추이 |
| :--- | :--- | :--- | :--- | :---: |
| 세미나실 B 빔프로젝터 | H01 (08-03, 2일) | H06 (09-02, 3일) | H16 (10-08, 4일) | 30일 -> 36일 |
| 체력단련실 러닝머신 1호 | H03 (08-17, 4일) | H11 (09-21, 6일) | H20 (10-21, 처리중) | 35일 -> 30일 (단축) |
| 대대 강당 무선 마이크 | H05 (08-31, 2일) | H13 (09-28, 3일) | H21 (10-22, 접수) | 28일 -> 24일 (단축) |

#### 2. 설비별 현장 추가 점검 체크리스트 (원인 예단 금지)
| 설비명 | 섣부른 추측 (지양) | 현장 객관적 점검 사항 (지향) | 담당 부서 |
| :--- | :--- | :--- | :---: |
| 빔프로젝터 | "기기가 낡아서 고장남" | 1. 케이블 단자 접촉 상태<br>2. 램프 사용 시간 누적량<br>3. 전원선 과열 여부 | 군수과 |
| 러닝머신 1호 | "장병들이 험하게 씀" | 1. 벨트 좌우 장력 균형<br>2. 모터 윤활유(오일) 상태<br>3. 롤러 베어링 마모도 | 군수과 |
| 무선 마이크 | "배터리 충전 불량" | 1. 수신기 안테나 각도/위치<br>2. 주파수 혼선 여부<br>3. 송신기 캡슐 단선 여부 | 정보통신실 |

- **심화 분석**: 대대 강당 무선 마이크는 재발 주기가 28일에서 24일로 가속화되고 있으므로, 단순 수리를 넘어 주파수 재설정 및 전면 기기 점검이 필요함.
"""

print(f"미션 06 실습 기록 완료: 프롬프트 {len(mission_06_prompt.strip())}자, 결과 {len(mission_06_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션6] 2회 이상 반복 고장 설비 탐색 및 재발 주기 계산
trouble_df = df_maintenance[df_maintenance['type'] == '고장신고']
freq = trouble_df.groupby(['facility', 'content']).size().reset_index(name='count')
recurring = freq[freq['count'] >= 2]

print("=== 2회 이상 반복 고장 설비 ===")
print(recurring)
